# State-space design: pole placement, LQR, observers and LQG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/topics/classical_control/state_space_design.ipynb)

One plant, the inverted pendulum measured through its angle alone, taken through the state-space design chapter with the [minilink](https://github.com/alx87grd/minilink) API:

1. the linear model about the upright, $\dot x = A x + B u$, $y = C x$, and its controllability and observability;
2. pole placement: the gain $K$ of $u = -K x$ that puts the eigenvalues of $A - BK$ where we ask;
3. LQR: the gain that minimizes $\int x^T Q x + u^T R u \, dt$;
4. observers that rebuild the state from the angle, $\dot{\hat x} = A \hat x + B u + L (y - C \hat x)$: the Luenberger and Kalman gains;
5. LQG: the LQR gain on the Kalman estimate, its poles (the separation principle), and the loop under noise;
6. the reference gain $N$ that makes the angle track a setpoint.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import (
    DiagramSystem,
    Gain,
    InvertedPendulum,
    OutputReference,
    Step,
    WhiteNoise,
    closed_loop,
    controllability,
    kalman,
    linearize,
    lqg,
    lqr,
    luenberger,
    observability,
    place,
)

X_BAR = np.array([0.0, 0.0])  # the operating point: upright, at rest
X0 = np.array([0.4, 0.0])  # every simulation starts 0.4 rad off the upright

## 1. The plant and its linear model

The catalog's `InvertedPendulum` has $\theta = 0$ upright and the torque $u$ as its input. The sensor is part of the plant: a `Gain` in series keeps the angle alone, so the measured output is $y = \theta$. `linearize` returns the linear model about the upright, $\dot x = A x + B u$, $y = C x$.

In [ ]:
pendulum = InvertedPendulum()
pendulum.x0 = X0
plant = pendulum >> Gain([[1.0, 0.0]])  # the sensor: y = θ

model = linearize(plant, X_BAR)
A, B, C = model.A(), model.B(), model.C()
print(model)

One eigenvalue of $A$ is positive: the upright falls on its own.

In [ ]:
np.linalg.eigvals(A)

In [ ]:
plant.plot_pzmap()

Every pole can be moved when the controllability matrix $\mathcal{C} = [B, AB]$ has full rank, and the state can be rebuilt from $y$ when the observability matrix $\mathcal{O} = [C; CA]$ has: both ranks must be $n = 2$.

In [ ]:
controllability(model).rank, observability(model).rank

## 2. Pole placement

State feedback $u = -K x$ gives the closed loop $\dot x = (A - BK) x$. With one input, Ackermann's formula returns the one $K$ that puts the eigenvalues of $A - BK$ where we ask. `place` designs on the plant (it linearizes it first) and returns the controller block; `@` closes the loop on the state.

In [ ]:
POLES = [-3.0 + 3.0j, -3.0 - 3.0j]

placed = place(plant, POLES, x_bar=X_BAR)
placed_loop = placed @ plant
placed_loop.plot_diagram()


In [ ]:
trajectory = placed_loop.compute_trajectory(tf=4.0)
placed.params["K"]

In [ ]:
placed_loop.animate()

In [ ]:
placed_loop.plot_trajectory()

In [ ]:
placed_loop.plot_pzmap()

## 3. LQR

Rather than choosing the poles, choose what the motion costs. The LQR gain minimizes $J = \int_0^\infty x^T Q x + u^T R u \, dt$: it is $K = R^{-1} B^T P$, with $P$ the stabilizing solution of the algebraic Riccati equation $A^T P + P A - P B R^{-1} B^T P + Q = 0$.

In [ ]:
Q = np.diag([10.0, 1.0])  # weights on θ and dθ/dt
R = np.array([[1.0]])  # weight on the torque

optimal = lqr(plant, Q, R, x_bar=X_BAR)
optimal_loop = optimal @ plant
optimal_loop.plot_diagram()

In [ ]:
trajectory = optimal_loop.compute_trajectory(tf=4.0)
optimal.params["K"]

In [ ]:
optimal_loop.plot_trajectory()

In [ ]:
optimal_loop.animate()

## 4. Observers: the state from the angle alone

Both gains above need the whole state $x = [\theta, \dot\theta]$; the sensor gives $\theta$ only. An observer runs the model and corrects it with what it mispredicts:

$$\dot{\hat x} = A \hat x + B u + L (y - C \hat x).$$

The estimation error $e = x - \hat x$ follows $\dot e = (A - LC)\, e$, so $L$ sets its poles. Choosing $L$ is the dual of choosing $K$: the eigenvalues of $A - LC$ are those of $A^T - C^T L^T$, the state-feedback problem on the pair $(A^T, C^T)$. `luenberger` places them; `kalman` makes them optimal for white noise.

In [ ]:
luenberger_observer = luenberger(plant, [-6.0, -7.0], x_bar=X_BAR)
print(luenberger_observer)

The Kalman filter is designed for the noise it will meet: a disturbance torque $d$ of intensity $W$, which enters like the command, so the process noise on the state is $Q_n = B W B^T$; and a noise of intensity $V$ on the measured angle, $R_n = V$. Its gain is $L = P C^T R_n^{-1}$, with $P$ the solution of the filter's Riccati equation, the control equation on the pair $(A^T, C^T)$.

In [ ]:
W = 1e-2  # intensity of the disturbance torque d
V = 1e-4  # intensity of the angle noise n
Q_N = W * B @ B.T
R_N = V * np.eye(1)

kalman_filter = kalman(plant, Q_N, R_N, x_bar=X_BAR)
print(kalman_filter)

To watch the filter work, run it beside the LQR loop of section 3: it reads the command $u$ and the measured angle $y$, and starts from $\hat x = 0$ while the pendulum starts at 0.4 rad. Its estimate $\hat\theta$, $\hat{\dot\theta}$ joins the true state within about two seconds, at the rate set by the filter's poles.

In [ ]:
watched = closed_loop(optimal, plant, reference=False)  # the LQR loop
watched.connect_new_output_port("controller", "u", "u")  # its command, for the filter

watch = DiagramSystem()
watch.add_subsystem(watched, "loop")
watch.add_subsystem(kalman_filter, "observer")
watch.connect("loop", "u", "observer", "u")
watch.connect("loop", "y", "observer", "y")
watch.plot_diagram()

In [ ]:
trajectory = watch.compute_trajectory(tf=3.0)

In [ ]:
watch.plot_trajectory(signals=((pendulum, "x"), (kalman_filter, "x")))

## 5. LQG: the gain on the estimate

The LQR gain acting on the Kalman estimate, $u = -K \hat x$, is the LQG compensator. `closed_loop(..., observer=)` wires the two by role: the filter reads the plant's measured $y$ and the command $u$, and the gain reads the estimate $\hat x$ in place of the state. The plant's own state is never read.

In [ ]:
lqg_loop = closed_loop(optimal, plant, observer=kalman_filter)

In [ ]:
lqg_loop.plot_diagram()

The separation principle: the loop's poles are the gain's and the filter's together, $\operatorname{eig}(A - BK) \cup \operatorname{eig}(A - LC)$, so the two designs can be made apart.

In [ ]:
K = optimal.params["K"]
L = kalman_filter.params["L"]

closed = np.linalg.eigvals(lqg_loop.linearize().A())
separate = np.concatenate([np.linalg.eigvals(A - B @ K), np.linalg.eigvals(A - L @ C)])
np.sort_complex(closed), np.sort_complex(separate)

`lqg` makes both designs in one call and returns them as one controller box, the filter under `observer` and the gain under `gain`, with $y$ in and $u$ out: `compensator @ plant` is the same loop.

In [ ]:
compensator = lqg(plant, Q, R, Q_N, R_N, x_bar=X_BAR)
print(compensator)
compensator.plot_diagram()

Now with the noise the filter was designed for. Given as source blocks, the disturbance torque of intensity $W$ and the angle noise of intensity $V$ are wired where they enter: this plant has no ports of its own for them, so a `Sum` adds $d$ to the torque and another adds $n$ to the measured angle before the filter reads it.

In [ ]:
noisy = closed_loop(
    optimal,
    plant,
    observer=kalman_filter,
    disturbance=WhiteNoise(1, psd=W, seed=1),
    noise=WhiteNoise(1, psd=V, seed=2),
)
noisy.plot_diagram()

In [ ]:
trajectory = noisy.compute_trajectory(tf=6.0)

In [ ]:
noisy.plot_trajectory()

In [ ]:
noisy.animate()

## 6. Tracking an angle: the reference gain N

To hold the pendulum at an angle $r$ rather than upright, feed the reference forward: $u = -K x + N r$. At steady state $\dot x = 0$ gives $y = -C (A - BK)^{-1} B N r$, so $N = -\left(C (A - BK)^{-1} B\right)^{-1}$ makes $y = r$. `OutputReference` takes $K$ and $N$; the setpoint, a source block given as the loop's `reference`, steps from 0 to 0.2 rad at $t = 1$ s.

In [ ]:
K = optimal.params["K"]
N = -np.linalg.inv(C @ np.linalg.inv(A - B @ K) @ B)
tracker = OutputReference(K, N, xbar=X_BAR)

tracking = closed_loop(tracker, plant, reference=Step(final_value=[0.2], step_time=1.0))
tracking.plot_diagram()

In [ ]:
trajectory = tracking.compute_trajectory(tf=6.0)

In [ ]:
tracking.plot_trajectory()

In [ ]:
tracking.animate()